# A2A Customer Support — KG-Guided Activation Steering
## Llama-2-7b-chat-hf Edition

### Model specs
| Setting | Value |
|---|---|
| `LOCAL_MODEL_NAME` | `Llama-2-7b-chat-hf` |
| `STEER_LAYER` | 20 |
| `STEER_ALPHA` | 15.0 |
| Hidden dim | 4096 |
| Total layers | 32 |
| VRAM required | ~14 GB bfloat16 |
| Prompt format | Chat `<s>[INST] <<SYS>>` |

### Runtime recommendation
**Google Colab T4 or A100 GPU** (14 GB fits on T4; A100 is comfortable).
`Runtime → Change runtime type → GPU`

### Files required
Place `main_7b.py` in the same directory as this notebook (or upload it to Colab).

### Secrets
```bash
export HF_TOKEN="hf_..."        # Hugging Face token (Llama-2 is gated)
export GROQ_API_KEY="gsk_..."   # Groq API key — free at https://console.groq.com
```


In [1]:
# ─── Cell 0a — AGGRESSIVE GPU CLEANUP (RUN THIS FIRST EVERY SESSION) ────────
import gc, sys

if 'main_module' in dir():
    try:
        main_module._model_cache.clear()
        print('  ✓ main_module._model_cache.clear() — done')
    except Exception as e:
        print(f'  main_module cache clear skipped: {e}')

_gone = [k for k in list(globals().keys())
         if any(x in k.lower() for x in ['model', 'tokenizer', 'main_module'])]
for _v in _gone:
    try: del globals()[_v]
    except: pass
if _gone:
    print(f'  ✓ Deleted globals: {_gone}')

_unloaded = [k for k in list(sys.modules.keys()) if 'main_module' in k]
for _m in _unloaded:
    del sys.modules[_m]
if _unloaded:
    print(f'  ✓ Unloaded from sys.modules: {_unloaded}')

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved  = torch.cuda.memory_reserved()  / 1e9
        total     = torch.cuda.get_device_properties(0).total_memory / 1e9
        free      = total - reserved
        print(f'  GPU         : {torch.cuda.get_device_name(0)}')
        print(f'  Total VRAM  : {total:.1f} GB')
        print(f'  Allocated   : {allocated:.2f} GB  ← should be ~0.0 GB')
        print(f'  Approx Free : {free:.1f} GB')
        if allocated > 1.0:
            print('  ⚠  VRAM still in use — Runtime → Restart runtime → re-run.')
        elif total < 14.0:
            print(f'  ⚠  Only {total:.1f} GB total VRAM. Llama-2-7b needs ~14 GB bfloat16.')
            print('  → Runtime → Change runtime type → T4 or A100 GPU')
        else:
            print('  ✓ VRAM clean and sufficient for Llama-2-7b-chat-hf.')
    else:
        print('  ⚠  No GPU! → Runtime → Change runtime type → GPU')
except ImportError:
    print('  torch not installed yet — run Cell 1')


  GPU         : NVIDIA RTX 6000 Ada Generation
  Total VRAM  : 50.9 GB
  Allocated   : 0.00 GB  ← should be ~0.0 GB
  Approx Free : 50.9 GB
  ✓ VRAM clean and sufficient for Llama-2-7b-chat-hf.


In [2]:
# ─── Cell 0b — MOUNT GOOGLE DRIVE FOR PERSISTENT MODEL CACHE ────────────────
# First run downloads ~14 GB; later runs load in ~1–2 min from Drive cache.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE = '/content/drive/MyDrive/hf_model_cache'
    os.makedirs(DRIVE_CACHE, exist_ok=True)
    os.environ['HF_HOME']               = DRIVE_CACHE
    os.environ['TRANSFORMERS_CACHE']    = DRIVE_CACHE
    os.environ['HF_DATASETS_CACHE']     = DRIVE_CACHE
    os.environ['HUGGINGFACE_HUB_CACHE'] = DRIVE_CACHE
    import glob
    existing = glob.glob(f'{DRIVE_CACHE}/**/*.safetensors', recursive=True)
    existing += glob.glob(f'{DRIVE_CACHE}/**/*.bin', recursive=True)
    print(f'  ✓ HuggingFace cache → {DRIVE_CACHE}')
    if existing:
        total_gb = sum(os.path.getsize(f) for f in existing) / 1e9
        print(f'  ✓ Found {len(existing)} cached weight file(s) ({total_gb:.1f} GB) — NO re-download!')
    else:
        print('  ℹ  No cached weights yet — first run downloads ~14 GB to Drive.')
except Exception as e:
    print(f'  Drive mount skipped ({e}) — using default HF cache.')


  Drive mount skipped (No module named 'google.colab') — using default HF cache.


In [3]:
# ─── Cell 1 — Install dependencies ──────────────────────────────────────────
%pip install -q \
    fastapi==0.115.0 \
    uvicorn==0.30.6 \
    httpx==0.27.2 \
    pydantic==2.8.2 \
    python-dotenv>=1.0.0 \
    langchain-groq>=0.1.9 \
    langchain-core>=0.2.0 \
    rich>=13.7.0 \
    transformers==4.40.0 \
    accelerate>=0.29.0 \
    huggingface_hub>=0.22.0 \
    scikit-learn>=1.4.0 \
    numpy>=1.26.0 \
    spacy

try:
    import torch
    print(f"✓ torch already available: {torch.__version__}")
except ImportError:
    !pip install -q "torch>=2.2.0"
    import torch
    print(f"✓ torch installed: {torch.__version__}")

!python -m spacy download en_core_web_sm
print("✓ All dependencies installed")


Note: you may need to restart the kernel to use updated packages.
✓ torch already available: 2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 33.2 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✓ All dependencies installed


In [4]:
# ─── Cell 1b — Verify spaCy NER model ───────────────────────────────────────
import spacy
nlp = spacy.load("en_core_web_sm")
doc = nlp("Priya Sharma ordered TechPro X200 Laptop under ORD-7741.")
print("✓ spaCy en_core_web_sm loaded")
for ent in doc.ents:
    print(f"    {ent.text!r:30s}  label={ent.label_}")


✓ spaCy en_core_web_sm loaded
    'Priya Sharma'                  label=PERSON
    'TechPro'                       label=ORG
    'X200'                          label=PRODUCT


In [ ]:
# ─── Cell 2 — Set secrets and authenticate with Hugging Face ─────────────────
#
# Option A (recommended): export HF_TOKEN and GROQ_API_KEY before launching.
# Option B: fill in the strings below (do NOT commit credentials to git).

import os
from huggingface_hub import login

# ── Uncomment and fill in if not already exported ────────────────────────────
os.environ["HF_TOKEN"]     = "HF_TOKEN"   # https://huggingface.co/settings/tokens
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"    # https://console.groq.com


HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise EnvironmentError(
        "HF_TOKEN not found.\n"
        "Run:  export HF_TOKEN='hf_...'\n"
        "Or uncomment the assignment line above."
    )

login(token=HF_TOKEN)
print("✓ Hugging Face login successful")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if GROQ_API_KEY:
    print("✓ GROQ_API_KEY found — full pipeline (Cell 12) will work")
else:
    print("⚠  GROQ_API_KEY not set — vector build/smoke-test will still work; Cell 12 requires it")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✓ Hugging Face login successful
✓ GROQ_API_KEY found — full pipeline (Cell 12) will work


In [6]:
# ─── Cell 3 — Verify GPU and versions ────────────────────────────────────────
import torch, transformers
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU          : {gpu_name}")
    print(f"VRAM         : {vram_gb:.1f} GB")
    if vram_gb < 14.0:
        print("⚠  Llama-2-7b-chat-hf needs ~14 GB VRAM in bfloat16. Use T4 or A100.")
    else:
        print("✓ Sufficient VRAM for Llama-2-7b-chat-hf in bfloat16")
else:
    print("⚠  No GPU detected — Runtime → Change runtime type → GPU")


torch        : 2.10.0+cu128
transformers : 4.40.0
CUDA         : True
GPU          : NVIDIA RTX 6000 Ada Generation
VRAM         : 50.9 GB
✓ Sufficient VRAM for Llama-2-7b-chat-hf in bfloat16


In [7]:
# ─── Cell 4 — Verify main_7b.py exists ───────────────────────────────────────
from pathlib import Path
main_file = Path("main_7b.py")
if not main_file.exists():
    raise FileNotFoundError(
        "main_7b.py not found.\n"
        f"CWD: {Path.cwd()}\n"
        "Upload main_7b.py to Colab or copy it to this directory."
    )
print(f"✓ Found main_7b.py — {main_file.stat().st_size / 1024:.1f} KB")


✓ Found main_7b.py — 95.2 KB


In [8]:
# ─── Cell 5 — Configure environment and import main_7b.py ───────────────────
# Parameters tuned for Llama-2-7b-chat-hf:
#   STEER_LAYER = 20  (≈ 63% depth in 32-layer model)
#   STEER_ALPHA = 15.0 (7b is more sensitive than 13b per unit alpha)
import os, importlib.util
from pathlib import Path

os.environ.setdefault("LOCAL_MODEL_NAME", "meta-llama/Llama-2-7b-chat-hf")
os.environ.setdefault("STEER_LAYER",      "20")
os.environ.setdefault("STEER_ALPHA",      "15.0")
os.environ.setdefault("STYLE_CACHE_DIR",  ".style_cache_7b")

Path(".style_cache_7b").mkdir(exist_ok=True)

spec        = importlib.util.spec_from_file_location("main_module", "main_7b.py")
main_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(main_module)

print("✓ main_7b.py imported successfully")
print("=" * 55)
print(f"  Model          : {os.environ['LOCAL_MODEL_NAME']}")
print(f"  Steer layer    : {os.environ['STEER_LAYER']}")
print(f"  Steer alpha    : {os.environ['STEER_ALPHA']}")
print(f"  Cache dir      : {os.environ['STYLE_CACHE_DIR']}")
print(f"  Contrast pairs : empathetic={len(main_module.CONTRAST_PAIRS['empathetic'])}, "
      f"formal={len(main_module.CONTRAST_PAIRS['formal'])}")
print("=" * 55)

nlp_check = main_module._get_nlp()
if nlp_check:
    print("✓ spaCy en_core_web_sm active for KG-NER")
else:
    print("⚠  spaCy unavailable — regex-only NER fallback active")


✓ main_7b.py imported successfully
  Model          : meta-llama/Llama-2-7b-chat-hf
  Steer layer    : 20
  Steer alpha    : 15.0
  Cache dir      : .style_cache_7b
  Contrast pairs : empathetic=16, formal=16
✓ spaCy en_core_web_sm active for KG-NER


In [9]:
# ─── Cell 6 — Load Llama-2-7b-chat-hf ONCE (GPU-first) ─────────────────────
# 7b requires ~14 GB bfloat16 — fits on a T4 (16 GB) with ~2 GB headroom.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = os.environ["LOCAL_MODEL_NAME"]
HF_TOKEN   = os.environ["HF_TOKEN"]

MODEL_NEEDS_GB = 14.0

if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    free_vram_gb  = (torch.cuda.get_device_properties(0).total_memory
                     - torch.cuda.memory_reserved()) / 1e9
    print(f'  GPU total VRAM  : {total_vram_gb:.1f} GB')
    print(f'  VRAM free now   : {free_vram_gb:.1f} GB')
else:
    total_vram_gb = free_vram_gb = 0

if free_vram_gb >= MODEL_NEEDS_GB:
    device_map_arg = {'': 0}
    max_memory_arg = None
    print('  Strategy : GPU-only (device_map={"":0}) — no CPU offload')
elif total_vram_gb >= MODEL_NEEDS_GB:
    gpu_limit = f'{int(total_vram_gb - 1)}GiB'
    device_map_arg = 'auto'
    max_memory_arg = {0: gpu_limit, 'cpu': '4GiB'}
    print(f'  Strategy        : auto + max_memory={{GPU: {gpu_limit}, cpu: 4GiB}}')
else:
    print(f'  ⚠  Only {total_vram_gb:.1f} GB VRAM. Proceeding with auto (will be slow).')
    device_map_arg = 'auto'
    max_memory_arg = None

print(f'Loading {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_load_kwargs = dict(trust_remote_code=True, torch_dtype=torch.bfloat16,
                    low_cpu_mem_usage=True, device_map=device_map_arg, token=HF_TOKEN)
if max_memory_arg:
    _load_kwargs['max_memory'] = max_memory_arg

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **_load_kwargs)
model.eval()

main_module._model_cache['model']     = model
main_module._model_cache['tokenizer'] = tokenizer

if torch.cuda.is_available():
    vram_used = torch.cuda.memory_allocated() / 1e9
    print(f'\n✓ {MODEL_NAME} loaded')
    print(f'  VRAM allocated  : {vram_used:.2f} GB')
    cpu_params = sum(1 for p in model.parameters() if p.device.type in ('cpu', 'meta'))
    gpu_params = sum(1 for p in model.parameters() if p.device.type == 'cuda')
    print(f'  GPU params      : {gpu_params}')
    if cpu_params > 0:
        print(f'  ⚠  CPU/meta params : {cpu_params}')
    else:
        print('  ✓ All parameters on GPU')

n_layers = len(model.model.layers) if hasattr(model, 'model') else '?'
hidden   = model.config.hidden_size if hasattr(model, 'config') else '?'
print(f'  Layers          : {n_layers}  (STEER_LAYER={os.environ["STEER_LAYER"]})')
print(f'  Hidden dim      : {hidden}  (expected 4096 for 7b)')


  GPU total VRAM  : 50.9 GB
  VRAM free now   : 50.9 GB
  Strategy : GPU-only (device_map={"":0}) — no CPU offload
Loading meta-llama/Llama-2-7b-chat-hf ...


/home/pakdd/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/pakdd/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]


✓ meta-llama/Llama-2-7b-chat-hf loaded
  VRAM allocated  : 13.54 GB
  GPU params      : 291
  ✓ All parameters on GPU
  Layers          : 32  (STEER_LAYER=20)
  Hidden dim      : 4096  (expected 4096 for 7b)


In [10]:
# ─── Cell 6b — Patch activation steering for device_map='auto' ──────────────
# Same device-safe pattern as the 13b notebook.
import torch

def _patched_get_activation(model, tokenizer, text: str):
    captured = {}
    def _hook(module, inp, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.device.type != 'meta':
            captured["act"] = hidden.detach().cpu().float()
    layer  = main_module._get_layer(model)
    handle = layer.register_forward_hook(_hook)
    try:
        real_device = next(
            (p.device for p in model.parameters() if p.device.type != 'meta'),
            torch.device('cpu')
        )
        inputs = tokenizer(text, return_tensors="pt", padding=False,
                           return_attention_mask=True).to(real_device)
        with torch.no_grad():
            model(**inputs)
    finally:
        handle.remove()
    act = captured.get("act")
    if act is None:
        raise RuntimeError("Hook did not capture activations — all params on meta device?")
    return act[0, -1, :]

def _patched_run_kg_activation_steering(defactualized_prompt: str, style: str, kg) -> str:
    steer_vec = main_module.get_style_vector(style)
    if steer_vec is None:
        raise RuntimeError(f"No style vector for '{style}'. Build it in Cells 8-9 first.")
    model_, tokenizer_ = main_module._get_model_and_tokenizer()
    real_device = next(
        (p.device for p in model_.parameters() if p.device.type != 'meta'),
        torch.device('cuda:0')
    )
    vec = steer_vec.to(device=real_device, dtype=torch.float32)
    style_prefix = {
        "empathetic": "I'm truly sorry to hear about this and I completely understand how frustrated you must be feeling right now.",
        "formal": "We acknowledge receipt of your complaint and wish to advise you that your case has been logged and assigned for review.",
    }.get(style, "Thank you for contacting us regarding this matter.")
    system_instruction = (
        f"You are a professional customer support agent. "
        f"Write a {style.upper()} support reply. "
        "Do NOT use headers or bullet points. "
        "Write ONLY the reply — 3 to 4 sentences maximum. "
        "Do NOT analyse the message. Just write the reply."
    )
    full_prompt = (
        f"<s>[INST] <<SYS>>\n{system_instruction}\n<</SYS>>\n\n"
        f"Customer message: {defactualized_prompt} [/INST] {style_prefix}"
    )
    inputs    = tokenizer_(full_prompt, return_tensors="pt", padding=False,
                            return_attention_mask=True).to(real_device)
    input_ids = inputs.input_ids
    target_layer = main_module._get_layer(model_)
    alpha = main_module.STEER_ALPHA

    def _steer_hook(module, inp, output):
        hidden = output[0] if isinstance(output, tuple) else output
        if hidden.device.type == 'meta': return output
        h_f32  = hidden.float()
        v      = vec.to(device=h_f32.device, dtype=torch.float32)
        h_f32  = h_f32 + alpha * v
        steered = h_f32.to(hidden.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered

    handle = target_layer.register_forward_hook(_steer_hook)
    try:
        with torch.no_grad():
            output_ids = model_.generate(
                input_ids, attention_mask=inputs.attention_mask,
                max_new_tokens=120, do_sample=False,
                pad_token_id=tokenizer_.eos_token_id,
                repetition_penalty=1.3, eos_token_id=tokenizer_.eos_token_id,
            )
    finally:
        handle.remove()
    new_ids   = output_ids[0][input_ids.shape[1]:]
    result    = tokenizer_.decode(new_ids, skip_special_tokens=True).strip()
    sentences = [s.strip() for s in result.split('.') if s.strip()]
    if len(sentences) > 4:
        result = '. '.join(sentences[:4]) + '.'
    return result

# Apply patches
main_module._get_activation             = _patched_get_activation
main_module._run_kg_activation_steering = _patched_run_kg_activation_steering
print("✓ Activation steering patched for device_map='auto' compatibility")
print(f"  STEER_ALPHA = {main_module.STEER_ALPHA} (7b tuning)")
print(f"  STEER_LAYER = {main_module.STEER_LAYER} (layer 20 of 32)")


✓ Activation steering patched for device_map='auto' compatibility
  STEER_ALPHA = 15.0 (7b tuning)
  STEER_LAYER = 20 (layer 20 of 32)


In [11]:
# ─── Cell 7 — Define build_vector_with_loaded_model ─────────────────────────
import torch
from pathlib import Path

def build_vector_with_loaded_model(style: str, method: str = "pca") -> torch.Tensor:
    """Build and cache a unit-norm style direction vector using the already-loaded 7b model."""
    pairs = main_module.CONTRAST_PAIRS.get(style)
    if not pairs:
        raise ValueError(f"No contrast pairs for '{style}'. Available: {list(main_module.CONTRAST_PAIRS)}")
    pos_acts, neg_acts = [], []
    for i, (pos_text, neg_text) in enumerate(pairs):
        print(f"  Processing pair {i+1:2d}/{len(pairs)} ...", end="\r")
        pos_acts.append(main_module._get_activation(model, tokenizer, pos_text))
        neg_acts.append(main_module._get_activation(model, tokenizer, neg_text))
    print(f"  {len(pairs)}/{len(pairs)} pairs processed              ")
    method_fns = {"pca": main_module._pca, "mean": main_module._mean_difference,
                  "logistic": main_module._logistic_regression}
    if method not in method_fns:
        raise ValueError(f"Unknown method '{method}'.")
    vec = method_fns[method](pos_acts, neg_acts)
    main_module._save_vector(vec, style, method)
    return vec

print("✓ build_vector_with_loaded_model() defined")


✓ build_vector_with_loaded_model() defined


In [12]:
# ─── Cell 8 — Build EMPATHETIC style vector ──────────────────────────────────
print("Building EMPATHETIC vector (16 contrast pairs × 2 activations) ...")
print("Expected time: ~1–2 min on T4 GPU")
print()
vec_emp = build_vector_with_loaded_model(style="empathetic", method="pca")
print(f"\n✓ EMPATHETIC vector built")
print(f"  shape : {list(vec_emp.shape)}")
print(f"  norm  : {vec_emp.norm().item():.6f}  (should be ~1.0)")
print(f"  dtype : {vec_emp.dtype}")


Building EMPATHETIC vector (16 contrast pairs × 2 activations) ...
Expected time: ~1–2 min on T4 GPU

  16/16 pairs processed              

✓ EMPATHETIC vector built
  shape : [4096]
  norm  : 1.000001  (should be ~1.0)
  dtype : torch.float32


In [13]:
# ─── Cell 9 — Build FORMAL style vector ─────────────────────────────────────
print("Building FORMAL vector (16 contrast pairs × 2 activations) ...")
print("Expected time: ~1–2 min on T4 GPU")
print()
vec_form = build_vector_with_loaded_model(style="formal", method="pca")
print(f"\n✓ FORMAL vector built")
print(f"  shape : {list(vec_form.shape)}")
print(f"  norm  : {vec_form.norm().item():.6f}  (should be ~1.0)")
print(f"  dtype : {vec_form.dtype}")


Building FORMAL vector (16 contrast pairs × 2 activations) ...
Expected time: ~1–2 min on T4 GPU

  16/16 pairs processed              

✓ FORMAL vector built
  shape : [4096]
  norm  : 1.000001  (should be ~1.0)
  dtype : torch.float32


In [14]:
# ─── Cell 10 — Verify both vectors ──────────────────────────────────────────
import torch
sim = torch.nn.functional.cosine_similarity(
    vec_emp.unsqueeze(0), vec_form.unsqueeze(0)
).item()
print("=" * 55)
print(f"  empathetic : norm={vec_emp.norm():.6f}   shape={list(vec_emp.shape)}")
print(f"  formal     : norm={vec_form.norm():.6f}   shape={list(vec_form.shape)}")
print(f"  cosine sim : {sim:.4f}")
print("=" * 55)

assert abs(vec_emp.norm().item()  - 1.0) < 0.01, f"empathetic not unit-norm! norm={vec_emp.norm():.4f}"
assert abs(vec_form.norm().item() - 1.0) < 0.01, f"formal not unit-norm! norm={vec_form.norm():.4f}"
assert vec_emp.shape[0]  == 4096, f"Expected hidden_dim=4096 for Llama-2-7b, got {vec_emp.shape[0]}"
assert vec_form.shape[0] == 4096, f"Expected hidden_dim=4096 for Llama-2-7b, got {vec_form.shape[0]}"
assert sim < 0.5, f"Cosine similarity {sim:.3f} too high — vectors may not be distinct!"

import os
cache_files = sorted(os.listdir(".style_cache_7b"))
print(f"\nCached files in .style_cache_7b/:")
for f in cache_files:
    size = os.path.getsize(f".style_cache_7b/{f}") / 1024
    print(f"  {f}  ({size:.1f} KB)")
print("  Note: ~16 KB each = 4096 float32 values (Llama-2-7b hidden dim)")
print()
print("✓ All assertions passed!")


  empathetic : norm=1.000001   shape=[4096]
  formal     : norm=1.000001   shape=[4096]
  cosine sim : -1.0000

Cached files in .style_cache_7b/:
  style_vec_empathetic_pca.pkl  (16.4 KB)
  style_vec_formal_pca.pkl  (16.4 KB)
  Note: ~16 KB each = 4096 float32 values (Llama-2-7b hidden dim)

✓ All assertions passed!


In [15]:
# ─── Cell 11 — Smoke test: activation steering changes tone ─────────────────
import torch
ALPHA_TEST = 15.0   # matches STEER_ALPHA for 7b in main_7b.py

TEST_PROMPT = (
    "<s>[INST] <<SYS>>\n"
    "You are a professional customer support agent. Write a support reply.\n"
    "<</SYS>>\n\n"
    "Customer message: My <PRODUCT> order <ORDER_ID> has an <ISSUE>. [/INST] "
)

def generate_steered(prompt: str, style_vec: torch.Tensor, alpha: float, label: str) -> str:
    real_device = next(
        (p.device for p in model.parameters() if p.device.type != 'meta'),
        torch.device('cuda:0')
    )
    layer = main_module._get_layer(model)
    def _hook(module, inp, output):
        h = output[0] if isinstance(output, tuple) else output
        if h.device.type == 'meta': return output
        h_f32 = h.float()
        v = style_vec.to(device=h_f32.device, dtype=torch.float32)
        h_f32 = h_f32 + alpha * v
        steered = h_f32.to(h.dtype)
        return (steered,) + output[1:] if isinstance(output, tuple) else steered
    handle = layer.register_forward_hook(_hook)
    inputs = tokenizer(prompt, return_tensors="pt").to(real_device)
    try:
        with torch.no_grad():
            out = model.generate(inputs.input_ids, max_new_tokens=80,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id,
                                 repetition_penalty=1.3)
    finally:
        handle.remove()
    new_ids = out[0][inputs.input_ids.shape[1]:]
    text    = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    print(f"\n[{label}]")
    print(f"  {text[:350]}")
    return text

print("=" * 60)
print("Smoke Test: same prompt — two style vectors — different tones")
print("=" * 60)
r_emp  = generate_steered(TEST_PROMPT, vec_emp,  ALPHA_TEST, "EMPATHETIC")
r_form = generate_steered(TEST_PROMPT, vec_form, ALPHA_TEST, "FORMAL")
print()
print("✓ Smoke test complete — confirm EMPATHETIC is warm, FORMAL is procedural.")


Smoke Test: same prompt — two style vectors — different tones


/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



[EMPATHETIC]
  Dear valued customer,

Thank you for reaching out to us regarding your concern with your recent purchase of our product (Order ID: <ORDER_ID>) . We apologize that you're experiencing issues with it and would be happy to assist in resolving the problem as soon as possible. Could you please provide more details about the issue you're facing? For exam

[FORMAL]
  Dear valued customer,

Thank you for reaching out to us regarding your concern with your recent purchase of our product. We apologize for any inconvenience this issue may have caused and would be happy to assist you in resolving it as soon as possible. Could you please provide more details about the problem you're experiencing? For example, what is

✓ Smoke test complete — confirm EMPATHETIC is warm, FORMAL is procedural.


In [16]:
# ─── Cell 12 — List saved .pkl files ────────────────────────────────────────
import pickle
from pathlib import Path

cache_dir = Path(".style_cache_7b")
pkl_files = sorted(cache_dir.glob("*.pkl"))
print("=" * 60)
print("Stored vectors in .style_cache_7b/")
print("=" * 60)
if not pkl_files:
    print("  (no .pkl files — did Cells 8 and 9 run successfully?)")
else:
    for pkl_path in pkl_files:
        size_kb = pkl_path.stat().st_size / 1024
        with open(pkl_path, "rb") as f:
            v = pickle.load(f)
        print(f"  {pkl_path.name:<45} {size_kb:6.2f} KB   "
              f"shape={list(v.shape)}  norm={v.norm():.4f}")
print("=" * 60)
print("  Note: ~16 KB each = 4096 float32 values (Llama-2-7b hidden dim)")
print()
print("Next steps:")
print("  • Rename .style_cache_7b → .style_cache in your production env")
print("  • Set STYLE_CACHE_DIR=.style_cache_7b in your .env file")
print("  • Set LOCAL_MODEL_NAME=meta-llama/Llama-2-7b-chat-hf in your .env")
print("  • python main_7b.py battery_issue   # single scenario")
print("  • python main_7b.py batch           # 100-case batch")


Stored vectors in .style_cache_7b/
  style_vec_empathetic_pca.pkl                   16.39 KB   shape=[4096]  norm=1.0000
  style_vec_formal_pca.pkl                       16.39 KB   shape=[4096]  norm=1.0000
  Note: ~16 KB each = 4096 float32 values (Llama-2-7b hidden dim)

Next steps:
  • Rename .style_cache_7b → .style_cache in your production env
  • Set STYLE_CACHE_DIR=.style_cache_7b in your .env file
  • Set LOCAL_MODEL_NAME=meta-llama/Llama-2-7b-chat-hf in your .env
  • python main_7b.py battery_issue   # single scenario
  • python main_7b.py batch           # 100-case batch


In [17]:
# ─── Cell 13 — [Optional] Run the full A2A pipeline (single scenario) ────────
# Requires GROQ_API_KEY.
# Scenarios: battery_issue | wrong_item | billing_error | delivery_delay
import asyncio

SCENARIO = "battery_issue"   # ← change to try other scenarios

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_YOUR_KEY_HERE":
    print("⚠  Skipping: GROQ_API_KEY not set. Fill it in Cell 2.")
else:
    from langchain_groq import ChatGroq
    llm          = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    llm_customer = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7)
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio; nest_asyncio.apply()
    print(f"Running pipeline — scenario: {SCENARIO}")
    record = asyncio.run(main_module.run_pipeline(SCENARIO, llm, llm_customer))
    emp_out = {"kg_summary": record["knowledge_graph"],
               "defactualized_prompt": record["empathetic_output"]["defactualized_prompt"]}
    main_module.console.print(main_module.make_kg_panel(emp_out))
    emp_response  = {**record["empathetic_output"], "kg_summary": record["knowledge_graph"]}
    form_response = {**record["formal_output"],     "kg_summary": record["knowledge_graph"]}
    main_module.console.print(main_module.make_comparison_table(emp_response, form_response))
    print(f"\nFull record keys: {list(record.keys())}")


Running pipeline — scenario: battery_issue


/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:492: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/pakdd/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:497: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out to support again, as the issue with my TechPro X200 Laptop (order number ORD-7741) persists   │
│ despite trying the suggested fix last week. The battery is still draining completely within 2 hours of a full   │
│ charge, which is really affecting my work - I have a tight deadline to meet tomorrow and I desperately need     │
│ this resolved today. I've already updated the drivers as advised, but the problem remains. Could you please     │
│ look into this urgently and provide a more permanent solution?"                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Can you tell me a bit more about when this started happening? Have │
│ there been any changes in your usage habits recently that could    │
│ help us identify the root cause of the issue? Any information you  │
│ can share will definitely help our team investigate this further   │
│ for you.                                                           │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Can you tell me a bit more about when this started happening? Have │
│ there been any changes in your usage habits recently that could    │
│ help us identify the root cause of the issue? Any information you  │
│ can                                                                │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our technical team will investigate the matter further and provide │
│ an update on possible solutions shortly. In the meantime, we       │
│ recommend trying a hard reset of the device to see if it resolves  │
│ the issue temporarily until a longer-term fix can be implemented.  │
│ Please let us know if there's anything else we can assist you with │
│ in the interim.                                                    │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our technical team will investigate the matter further and provide │
│ an update on possible solutions shortly. In the meantime, we       │
│ recommend trying a hard reset of the device to see if it resolves  │
│ the is                                                             │
╰────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Knowledge Graph ────────────────────────────────────────────────╮
│ Nodes (6):                                                                                                      │
│   (PRODUCT) TechPro X200 Laptop  (salience: 0.8)                                                                │
│   (ORDER_ID) ORD-7741  (salience: 0.9)                                                                          │
│   (ISSUE) my TechPro X200 Laptop (order number ORD-7741) persists despite trying the suggested fix last week    │
│ (salience: 0.9)                                                                                                 │
│   (URGENCY) high  (salience: 1.0)                                                                               │
│   (SENTIMENT) polite  (salience: 0.7)                                                                           │
│   (CUSTOMER_NAME) Priya Sharma  (salience: 0.7)                                                                 │
│                                                                                                                 │
│ Edges (4):                                                                                                      │
│   TechPro X200 Laptop       —has_issue→ my TechPro X200 Laptop (o                                               │
│   ORD-7741                  —about_product→ TechPro X200 Laptop                                                 │
│   Priya Sharma              —placed_order→ ORD-7741                                                             │
│   my TechPro X200 Laptop (o —has_urgency→ high                                                                  │
│                                                                                                                 │
│ Defactualized prompt:                                                                                           │
│   I'm reaching out to support again, as the issue with <ISSUE>. The battery is still draining completely within │
│ 2 hours of a full charge, which is really affecting my work - I have a tight deadline to m                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                   Style Comparison — Same Facts, Different Tone                                   
┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Dimension       ┃ Empathetic Pipeline                           ┃ Formal Pipeline                               ┃
┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Target style    │ empathetic                                    │ formal                                        │
│ Steering mode   │ kg_activation                                 │ kg_activation                                 │
│ KG nodes        │ 6                                             │ 6                                             │
│ KG edges        │ 4                                             │ 4                                             │
│ Response (first │ Can you tell me a bit more about when this    │ Our technical team will investigate the       │
│ 200 chars)      │ started happening? Have there been any        │ matter further and provide an update on       │
│                 │ changes in your usage habits recently that    │ possible solutions shortly. In the meantime,  │
│                 │ could help us identify the root cause of the  │ we recommend trying a hard reset of the       │
│                 │ issue? Any information you can                │ device to see if it resolves the is           │
└─────────────────┴───────────────────────────────────────────────┴───────────────────────────────────────────────┘


Full record keys: ['input', 'knowledge_graph', 'empathetic_output', 'formal_output']


In [18]:
# ─── Cell 14 — [Optional] Run the full 100-case batch ───────────────────────
# ⚠  ~15–40 min depending on Groq rate limits.
import asyncio
from pathlib import Path

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY or GROQ_API_KEY == "gsk_YOUR_KEY_HERE":
    print("⚠  Skipping: GROQ_API_KEY not set.")
else:
    Path("outputs_7b").mkdir(exist_ok=True)
    Path("outputs_7b/knowledge_graphs").mkdir(exist_ok=True)
    try:
        import nest_asyncio; nest_asyncio.apply()
    except ImportError:
        import subprocess; subprocess.run(["pip", "install", "-q", "nest_asyncio"], check=True)
        import nest_asyncio; nest_asyncio.apply()
    print(f"Starting batch — {len(main_module.BATCH_CASES)} cases")
    asyncio.run(main_module.run_batch())


Starting batch — 100 cases


──────────────────────────────────── A2A Customer Support — 100-Case Batch Run ────────────────────────────────────

Output file : outputs/results_20260524_034756.jsonl
  Total cases : 100
  Activation steering: ENABLED

──────────────── Case   1/100  scenario=battery_issue  customer=Priya Sharma  sentiment=frustrated ────────────────

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Priya Sharma   Order: ORD-7741                                                                        │
│ Product: TechPro X200 Laptop                                                                                    │
│ Issue: battery draining completely within 2 hours of a full charge                                              │
│ Sentiment: FRUSTRATED   Urgency: high                                                                           │
│                                                                                                                 │
│ "I'm reaching out to support for the second time regarding my TechPro X200 Laptop, order number ORD-7741, which │
│ I purchased just three weeks ago. Despite updating the drivers as suggested during my previous support          │
│ interaction, the battery issue persists - it's still draining completely within 2 hours of a full charge, which │
│ is severely impacting my work. I have a critical deadline to meet tomorrow and I urgently need a solution to    │
│ this problem, so I would appreciate it if we could resolve this today. Could you please assist me with a more   │
│ permanent fix or provide a suitable alternative?"                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── Customer Message ────────────────────────────────────────────────╮
│ Customer: Ximena Flores   Order: ORD-8858                                                                       │
│ Product: Ergonomic Office Chair                                                                                 │
│ Issue: delivery now 3 weeks overdue with no update from courier                                                 │
│ Sentiment: NEUTRAL   Urgency: medium                                                                            │
│                                                                                                                 │
│ "I'm reaching out about my order, ORD-8858, for the Ergonomic Office Chair, which was supposed to be delivered  │
│ three weeks ago. Unfortunately, I still haven't received it and haven't gotten any updates from the courier.    │
│ I've been working on the floor for weeks, so I'm eager to get this resolved - could you please provide a clear  │
│ timeline for when I can expect to receive my chair, or let me know if I can get a refund so I can look into     │
│ purchasing one locally? I'd appreciate any help you can provide in getting this sorted out."                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────── Empathetic — Support Response ───────────────────╮
│ Style: EMPATHETIC   Steering mode: kg_activation                   │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Can you please give us some more details regarding your delivery   │
│ status like tracking information or contact number of the courier  │
│ company that might have caused the delay? We will do our best to   │
│ resolve this issue as soon possible and ensure that you received   │
│ your product within the next few days.                             │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Can you please give us some more details regarding your delivery   │
│ status like tracking information or contact number of the courier  │
│ company that might have caused the delay? We will do our best to   │
│ reso                                                               │
╰────────────────────────────────────────────────────────────────────╯
╭──────────────────── Formal — Support Response ─────────────────────╮
│ Style: FORMAL   Steering mode: kg_activation                       │
│                                                                    │
│ Final response (rehydrated):                                       │
│ Our team is currently investigating the matter regarding the delay │
│ in delivery of your product with Order ID (insert here). Please    │
│ allow us some time to gather more information before we proceed    │
│ further. You will hear back from our team within the next two      │
│ business days with an update on the status of your request. Thank  │
│ you for choosing our brand!                                        │
│                                                                    │
│ Defactualized output (pre-rehydration):                            │
│ Our team is currently investigating the matter regarding the delay │
│ in delivery of your product with Order ID (insert here). Please    │
│ allow us some time to gather more information before we proceed    │
│ furth                                                              │
╰────────────────────────────────────────────────────────────────────╯

✓ Saved case 100  |  KG → knowledge_graphs/kg_100_delivery_delay_ORD-8858_Ximena_Flores.json

───────────────────────────────────────────────── Batch complete ──────────────────────────────────────────────────

Success: 100/100
  Failed:  0/100
  Results: outputs/results_20260524_034756.jsonl
  KGs:     outputs/knowledge_graphs/  (100 files)